# Step 10 — Score every classifier against the same ground truth

**Inputs:** `09_ground_truth.parquet`, the frozen buildings file, and any
`06b_llm_predictions_*.parquet`
**Outputs:** `10_score_detail_{arm}.csv`, `10_score_summary_{arm}.csv`,
`10_score_comparison.csv`, `10_rule_diagnostics.csv`

Every arm is scored over **one `src` frame, one truth table, one set of
`validation_utils` calls**. That is what makes the numbers comparable — not that the
arms ran the same preprocessing code, but that they are scored on the same rows through
the same functions.

## Read this before quoting any number

### 1. The workbook is partly the LLM's own output

The spreadsheet arrived **pre-filled with an earlier LLM run's predictions**; a green
cell means a human read that answer and affirmed it. From the ground truth itself:
**553 of 882** activity rows (62.7%) and **719 of 874** Bosserhof rows (82.3%) are
`human confirmed the …`. On those rows the "truth" *is* an LLM answer.

So the subsets are **bounds, not estimates**:

| subset | for the LLM | for the rule engine |
|---|---|---|
| green | **upper bound** — truth is a prior LLM answer | genuine held-out estimate |
| red | **lower bound** — selected *because* a prior LLM got them wrong | genuine, harder subset |
| all | mixture of the two; least interpretable | genuine estimate |

Red is *not* the fair subset. It is adversarially selected against the LLM, so a rerun
scores roughly `1 − self-consistency` there. Report green and red as a bracket. The
prefill-agreement diagnostic at the end measures the self-consistency term directly, for
free.

### 2. The arms are not given the same evidence unless you make them

`rule_utils` says, verbatim: *"DELIBERATELY NOT USED: osm_names / any free-text business
name. … That is the accepted limitation being measured here."* 75.9% of scoreable
buildings carry a name. So `llm_full` vs `rule` measures **with names vs without names**.
`llm_blind` is the method comparison.

### 3. Trivial constants beat the rule engine on precision

`Workers` appears in 850 of 882 truth sets. A classifier that always answers exactly
`{Workers}` scores **precision 0.964** — above the rule engine's 0.845. Activity
precision is *anti-correlated with effort*: predict less, score higher. The constant arms
below cost nothing and exist so no precision figure is ever quoted without its floor.

In [1]:
import sys
sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))
from config import (
    VALIDATION_GROUND_TRUTH, VALIDATION_BUILDINGS_FILE, VALIDATION_SOURCE_FILE,
    VALIDATION_SCORE_DETAIL, VALIDATION_SCORE_SUMMARY, VALIDATION_SCORE_COMPARISON,
    VALIDATION_RULE_DIAGNOSTICS, VALIDATION_DIR, score_paths,
    BUILDING_FUNCTION_CODELIST, ZONE_ACTIVITY_COLUMNS, GENERIC_COMMERCIAL,
)
from rule_utils import classify_building, reachable_bosserhof_classes
from validation_utils import (
    collapse_to_zone_activities, score_activities, score_bosserhof,
    per_activity_breakdown, format_activity_report, format_bosserhof_report,
    label_accounting, bosserhof_accounting, resolve_prediction_bosserhof,
    BOSSERHOF_KNOWN, classes_mentioned,
)
from llm_utils import normalise_mid_labels

import hashlib
import pandas as pd
import pyogrio

pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 55)


def as_set(value):
    # Coerce a label cell to a set. Parquet round-trips list columns as numpy
    # arrays and `value or []` raises "truth value of an array is ambiguous",
    # so the None check has to be explicit rather than relying on falsiness.
    return set() if value is None else set(value)


for path, what in [(VALIDATION_GROUND_TRUTH, 'notebook 09 output'),
                   (VALIDATION_BUILDINGS_FILE, 'the frozen buildings the sample was drawn from')]:
    if not path.exists():
        raise FileNotFoundError(f'{path}\n  missing: {what}')

# The frozen file is 353 MB, lives outside the repo and is not distributed. It is the
# ONLY input the workbook's positional ids can be scored against, so record which copy
# produced these numbers.
_h = hashlib.sha256()
with open(VALIDATION_BUILDINGS_FILE, 'rb') as fh:
    for block in iter(lambda: fh.read(1 << 22), b''):
        _h.update(block)
SRC_SHA256 = _h.hexdigest()
print(f'buildings file: {VALIDATION_BUILDINGS_FILE.name}')
print(f'  sha256: {SRC_SHA256}')
print('Inputs present.')

C:\Users\Mayur Patel\anaconda3\envs\GNNs\Lib\site-packages\pyogrio\core.py:34: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


buildings file: condensed_buildings_with_pois.gpkg
  sha256: 57e43e93693e023b8e9f5a56931654a1e002036c56dcc41d2f527b174503fd2e
Inputs present.


---
## Step 1 — Ground truth

Reused as committed, never regenerated. Notebook 09's activity-typo guard raises by
design on unrecognised tokens, and `config.py` warns against pre-empting it — a
regenerated truth table risks differing from the one the published rule-engine baseline
was measured against, which would silently destroy comparability between the arms.

In [2]:
truth = pd.read_parquet(VALIDATION_GROUND_TRUTH)
truth['gml_id'] = truth['gml_id'].astype(str)

_leaked = [c for c in truth.columns
           if any(t in c.lower() for t in ('predicted', 'prefill', 'replacement',
                                           'precision', 'recall', 'accuracy'))]
assert not _leaked, f'ground truth contains prediction or metric columns: {_leaked}'

N_ACT = int(truth['activities_scoreable'].sum())
N_BOSS = int(truth['bosserhof_scoreable'].sum())
print(f'{len(truth):,} annotated buildings')
print(f'  activities scoreable : {N_ACT:,}')
print(f'  bosserhof  scoreable : {N_BOSS:,}')

# The circularity, measured rather than asserted.
ACT_GREEN = 'human confirmed the label'
BOSS_GREEN = 'human confirmed the class'
n_ag = int((truth['activities_scoreable_reason'] == ACT_GREEN).sum())
n_bg = int((truth['bosserhof_scoreable_reason'] == BOSS_GREEN).sum())
print(f'\nCIRCULARITY (truth = an earlier LLM answer a human affirmed):')
print(f'  activities: {n_ag:,} of {N_ACT:,} ({n_ag / N_ACT:.1%}) green')
print(f'  bosserhof : {n_bg:,} of {N_BOSS:,} ({n_bg / N_BOSS:.1%}) green')
print(f'  red (human typed the answer): activities {N_ACT - n_ag:,}, bosserhof {N_BOSS - n_bg:,}')

1,391 annotated buildings
  activities scoreable : 882
  bosserhof  scoreable : 874

CIRCULARITY (truth = an earlier LLM answer a human affirmed):
  activities: 553 of 882 (62.7%) green
  bosserhof : 719 of 874 (82.3%) green
  red (human typed the answer): activities 329, bosserhof 155


---
## Step 2 — The buildings, loaded once

`NEEDED` is the **union** of what the rule engine reads and what the LLM prompt reads —
16 columns, not the 12 the rule engine alone needs. The loader keeps only columns that
exist, so widening it is harmless, and every one of the 16 is present in the frozen file.

The frozen file has no `function` column, so the ALKIS code is recovered from `label_en`
behind a one-to-one assertion. That recovery is a **documented asymmetry**: in production
the authoritative code is carried through. It is one-to-one over the 71 labels present
here, but it is not the same code path.

In [3]:
NEEDED = [
    # identity + rule engine
    'gml_id', 'volume_m3', 'label_en', 'amenity', 'shop', 'tourism', 'building',
    'information', 'additional_information', 'osm_building_type',
    'osm_landuse_class', 'osm_names',
    # additionally read by llm_utils.row_to_llm_input
    'website', 'email', 'tags_search', 'gfk_class', 'ALKIS_Landuse_info',
    'osm_landuse_name',
]

fields = set(pyogrio.read_info(VALIDATION_BUILDINGS_FILE)['fields'])
missing = [c for c in NEEDED if c not in fields]
cols = [c for c in NEEDED if c in fields]
src = pyogrio.read_dataframe(VALIDATION_BUILDINGS_FILE, columns=cols, read_geometry=False)
src['gml_id'] = src['gml_id'].astype(str)
print(f'{VALIDATION_BUILDINGS_FILE.name}: {len(src):,} buildings, {len(cols)} of {len(NEEDED)} columns')
if missing:
    print(f'  absent (LLM prompt would have been starved of these): {missing}')

# --- recover the ALKIS code from the English label, one-to-one asserted ---
codelist = pd.read_csv(BUILDING_FUNCTION_CODELIST, encoding='utf-8', encoding_errors='replace')
per_label = codelist.groupby('label_en')['function'].nunique()
ambiguous = set(per_label[per_label > 1].index)

present = set(src['label_en'].dropna().unique())
unknown = present - set(codelist['label_en'])
collides = present & ambiguous
if unknown:
    raise AssertionError(f'{len(unknown)} label_en values absent from the codelist: '
                         f'{sorted(unknown)[:5]}')
if collides:
    raise AssertionError(
        'label_en to function is NOT one-to-one here; these labels are shared by more '
        f'than one ALKIS code, so keying on them would mis-classify: {sorted(collides)}.')
print(f'  label_en to function: one-to-one verified over {len(present)} labels')
src['function'] = src['label_en'].map(dict(zip(codelist['label_en'], codelist['function'])))

condensed_buildings_with_pois.gpkg: 578,080 buildings, 18 of 18 columns
  label_en to function: one-to-one verified over 71 labels


---
## Step 3 — The contract every arm satisfies

One normaliser for all arms, so no arm can win or lose on formatting.

`bosserhof_class` must end as a canonical string or literal `None` — **never `NaN`**.
`score_bosserhof` maps `None` to `''` (matching the 14 truth-`''` rows) but maps float
`NaN` to the string `'nan'`, which matches nothing. `classify_building` returns `None`;
parquet hands back `NaN`. Without the coercion the LLM arm loses those rows for a dtype
reason.

`present` replaces `interpreted_type.notna()` as the row-presence sentinel.
`interpreted_type` was doing two jobs — presence flag *and* a categorical of rule-layer
names that the tier table switches on. The LLM writes prose there, which satisfies
`notna()` but makes several tiers unreachable, producing a `tier` column that silently
lies.

In [4]:
def to_contract(pred, arm):
    # Normalise any arm's predictions to the one schema the scorer accepts.
    assert len(pred) > 0, f'{arm}: empty prediction frame'
    pred = pred.copy()
    pred['gml_id'] = pred['gml_id'].astype(str)
    assert pred['gml_id'].is_unique, f'{arm}: duplicate gml_id'

    pred['mid_labels'] = pred['mid_labels'].map(normalise_mid_labels)
    pred['zone_activities'] = pred['mid_labels'].map(collapse_to_zone_activities)

    bc = pred['bosserhof_class'].map(resolve_prediction_bosserhof)
    pred['bosserhof_class'] = bc.where(bc.notna(), None)   # NaN -> None, never 'nan'
    pred['present'] = True

    keep = ['gml_id', 'mid_labels', 'zone_activities', 'bosserhof_class', 'present']
    if 'interpreted_type' in pred.columns:
        keep.append('interpreted_type')
    if 'src_file' in pred.columns:
        keep.append('src_file')
    return pred[keep].merge(src[['gml_id', 'volume_m3', 'label_en', 'function']],
                            on='gml_id', how='left', validate='one_to_one')


def rule_predictions():
    # Re-run the rule engine in-process over src — never read from a GeoPackage.
    # This is how the published baseline was produced, and it is what lets both
    # arms share the source rows without sharing a file.
    return pd.DataFrame([classify_building(r) for r in src.to_dict('records')])


def constant_predictions(labels, boss=None):
    # A classifier that ignores its input entirely. The interpretability floor.
    return pd.DataFrame({
        'gml_id': src['gml_id'],
        'mid_labels': [list(labels)] * len(src),
        'bosserhof_class': [boss] * len(src),
    })


arms = {}
arms['rule'] = to_contract(rule_predictions(), 'rule')
print(f"rule: {len(arms['rule']):,} classified")

MAJORITY_BOSS = 'retail small scale'
arms['const_workers'] = to_contract(constant_predictions(['work']), 'const_workers')
arms['const_workers_leisure'] = to_contract(
    constant_predictions(['work', 'leisure']), 'const_workers_leisure')
arms['const_boss_majority'] = to_contract(
    constant_predictions([], MAJORITY_BOSS), 'const_boss_majority')

# --- discover every LLM benchmark checkpoint ---
for path in sorted(VALIDATION_DIR.glob('06b_llm_predictions_*.parquet')):
    name = path.stem.replace('06b_llm_predictions_', 'llm_')
    frame = pd.read_parquet(path)
    if 'src_file' in frame.columns:
        bad = set(frame['src_file'].dropna().unique()) - {VALIDATION_BUILDINGS_FILE.name}
        assert not bad, (f'GUARD 0  {name}: predictions derived from {bad}, not from '
                         f'{VALIDATION_BUILDINGS_FILE.name}')
    arms[name] = to_contract(frame, name)
    print(f'{name}: {len(frame):,} predictions')

llm_arms = [a for a in arms if a.startswith('llm_')]
if not llm_arms:
    print('\nNo LLM checkpoints yet — scoring the rule and baseline arms only.')
print(f'\narms: {list(arms)}')

rule: 578,080 classified



No LLM checkpoints yet — scoring the rule and baseline arms only.

arms: ['rule', 'const_workers', 'const_workers_leisure', 'const_boss_majority']


---
## Step 4 — Identity guards

`gml_id` is a per-run positional index, so a healthy-looking join is not evidence the
right buildings are being scored. Volume agreement is.

Both guards now **raise**. The previous GUARD 1 was a bare print with no threshold, and
GUARD 2 read `if len(both) and share < 0.95` — which short-circuits when *zero* rows
join. A `gml_id` dtype mismatch (`'236928.0'`) therefore printed "IDENTITY CONFIRMED"
and returned precision 0.0 / recall 0.0, which reads as "the LLM is terrible" but is a
plumbing bug.

In [5]:
def check_identity(pred, arm):
    m = truth.merge(pred.rename(columns={'volume_m3': 'volume_m3_pred'}),
                    on='gml_id', how='left', validate='one_to_one')
    have = m['present'].fillna(False).astype(bool)

    # Denominator is the rows that CAN be scored, not the whole workbook. Notebook
    # 06b deliberately skips the 506 buildings with no ground truth on either
    # dimension — every arm drops them here anyway, so paying an API call for them
    # buys nothing. The rule arm classifies all 578,080 for free and so covers
    # them; requiring the same of a paid arm would just be waste.
    need = m['activities_scoreable'] | m['bosserhof_scoreable']
    n_need = int(need.sum())
    n_matched = int((have & need).sum())

    if n_matched < 0.95 * n_need:
        raise AssertionError(
            f'GUARD 1  {arm}: only {n_matched:,}/{n_need:,} SCOREABLE annotated ids are '
            'present in the predictions. Check the gml_id dtype on both sides before '
            'reading any metric below — a dtype mismatch joins zero rows and every '
            'metric downstream reads 0.0.')

    both = m[m['volume_m3'].notna() & m['volume_m3_pred'].notna()]
    if not len(both):
        raise AssertionError(
            f'GUARD 2  {arm}: no rows have a volume on both sides, so identity cannot be '
            'verified at all. This is what a total gml_id dtype mismatch looks like.')

    rel = ((both['volume_m3'] - both['volume_m3_pred']).abs()
           / both['volume_m3'].abs().clip(lower=1e-9))
    share = float((rel < 1e-3).mean())
    if share < 0.95:
        print(both.assign(rel=rel).nlargest(5, 'rel')[
            ['gml_id', 'volume_m3', 'volume_m3_pred', 'osm_names']].to_string(index=False))
        raise AssertionError(
            f'GUARD 2  {arm}: only {share:.1%} of gml_ids have a matching volume. The ids '
            'resolve but point at DIFFERENT buildings — the sample came from another run '
            'of notebook 05. Refusing to score.')

    print(f'  {arm:24s} scoreable ids {n_matched:,}/{n_need:,}  volumes {share:.2%}')
    return m


print('IDENTITY')
merged = {arm: check_identity(pred, arm) for arm, pred in arms.items()}
print('\nCONFIRMED - every arm is scored on the annotated buildings.')

IDENTITY


  rule                     scoreable ids 885/885  volumes 100.00%


  const_workers            scoreable ids 885/885  volumes 100.00%


  const_workers_leisure    scoreable ids 885/885  volumes 100.00%


  const_boss_majority      scoreable ids 885/885  volumes 100.00%

CONFIRMED - every arm is scored on the annotated buildings.


---
## Step 5 — Score every arm on every subset

`workers_included=False` is a co-primary view, not a curiosity. `rule_utils` auto-adds
`work` to any building carrying a Bosserhof class — *"a Bosserhof class IS a
worker-density rating"*. `Workers` is 55.7% of all truth labels, so that one line
dominates recall: ablating it moves the rule engine from 0.845/0.785 to 0.760/0.438.
It is a labelling convention, not a classification skill, and reporting metrics with and
without it separates the two.

In [6]:
def act_pairs(m, subset, drop_workers):
    sel = m['activities_scoreable'] & m['present'].fillna(False).astype(bool)
    if subset == 'green':
        sel &= m['activities_scoreable_reason'] == ACT_GREEN
    elif subset == 'red':
        sel &= m['activities_scoreable_reason'] != ACT_GREEN
    rows = m[sel]
    out = []
    for _, r in rows.iterrows():
        p, t = set(r['zone_activities']), as_set(r['activities_truth'])
        if drop_workers:
            p, t = p - {'Workers'}, t - {'Workers'}
        out.append((r['gml_id'], p, t))
    return out


def boss_pairs(m, subset):
    sel = m['bosserhof_scoreable'] & m['present'].fillna(False).astype(bool)
    if subset == 'green':
        sel &= m['bosserhof_scoreable_reason'] == BOSS_GREEN
    elif subset == 'red':
        sel &= m['bosserhof_scoreable_reason'] != BOSS_GREEN
    rows = m[sel]
    return [(r['gml_id'], r['bosserhof_class'], r['bosserhof_truth'])
            for _, r in rows.iterrows()]


records = []
for arm, m in merged.items():
    for subset in ('all', 'green', 'red'):
        for drop_w in (False, True):
            acc = label_accounting(act_pairs(m, subset, drop_w))
            records.append({'arm': arm, 'dimension': 'activities', 'subset': subset,
                            'workers_included': not drop_w, **acc})
        bacc = bosserhof_accounting(boss_pairs(m, subset))
        records.append({'arm': arm, 'dimension': 'bosserhof', 'subset': subset,
                        'workers_included': True, 'buildings': bacc['buildings'],
                        'correct': bacc['correct'], 'accuracy': bacc['accuracy'],
                        'accuracy_lo': bacc['accuracy_lo'], 'accuracy_hi': bacc['accuracy_hi']})

comparison = pd.DataFrame(records)

# Equal denominators are the precondition for comparing arms at all. This one assertion
# also catches LLM coverage shrinkage, which would otherwise pass unnoticed.
for (dim, sub, wi), grp in comparison.groupby(['dimension', 'subset', 'workers_included']):
    ns = grp.set_index('arm')['buildings']
    assert ns.nunique() == 1, (
        f'{dim}/{sub}: arms scored over DIFFERENT numbers of buildings, so their metrics '
        f'are not comparable:\n{ns.to_string()}')
print('Denominators equal across arms for every (dimension, subset).\n')

VALIDATION_SCORE_COMPARISON.parent.mkdir(parents=True, exist_ok=True)
comparison.to_csv(VALIDATION_SCORE_COMPARISON, index=False)
print(f'-> {VALIDATION_SCORE_COMPARISON.name}')

view = comparison[(comparison.dimension == 'activities') & comparison.workers_included]
print('\nACTIVITIES (Workers included)')
print(view.pivot(index='arm', columns='subset',
                 values=['precision', 'recall', 'exact_set_match']).round(3).to_string())
print('\nBOSSERHOF')
bview = comparison[comparison.dimension == 'bosserhof']
print(bview.pivot(index='arm', columns='subset', values='accuracy').round(3).to_string())

Denominators equal across arms for every (dimension, subset).

-> 10_score_comparison.csv

ACTIVITIES (Workers included)
                      precision               recall               exact_set_match              
subset                      all  green    red    all  green    red             all  green    red
arm                                                                                             
const_boss_majority       0.000  0.000  0.000  0.000  0.000  0.000           0.015  0.000  0.040
const_workers             0.964  0.973  0.948  0.557  0.579  0.523           0.313  0.347  0.255
const_workers_leisure     0.666  0.646  0.698  0.769  0.770  0.769           0.286  0.253  0.340
rule                      0.845  0.929  0.716  0.785  0.859  0.670           0.543  0.725  0.237

BOSSERHOF
subset                   all  green    red
arm                                       
const_boss_majority    0.129  0.152  0.026
const_workers          0.016  0.000  0.090
const_workers_lei

---
## Step 6 — Rule-engine detail, and the regression gate

The rule arm's headline figures must reproduce the published baseline exactly:
`882 / 0.8449 / 0.7851 / 0.5431` and Bosserhof `874 / 0.5034 / 4 not producible`.
Anything else means this refactor changed the maths, and the head-to-head would be
measuring the refactor.

In [7]:
rule_m = merged['rule']
rule_act = act_pairs(rule_m, 'all', drop_workers=False)
rule_metrics, rule_rows = score_activities(rule_act)
print(format_activity_report(rule_metrics, 'RULE ENGINE - ACTIVITIES'))

print()
breakdown = pd.DataFrame(per_activity_breakdown(rule_act))
breakdown['miss_rate'] = (breakdown['missed']
                          / breakdown['in_truth'].where(breakdown['in_truth'] > 0)).round(3)
print(breakdown.sort_values('in_truth', ascending=False).to_string(index=False))

rule_boss = boss_pairs(rule_m, 'all')
rule_boss_metrics, rule_boss_rows = score_bosserhof(rule_boss)
print()
print(format_bosserhof_report(rule_boss_metrics, 'RULE ENGINE - BOSSERHOF'))

# --- regression gate against the frozen published baseline ---
BASELINE = {('activities', 'buildings_scored'): 882, ('activities', 'precision'): 0.8449,
            ('activities', 'recall'): 0.7851, ('activities', 'exact_set_match'): 0.5431,
            ('bosserhof', 'buildings_scored'): 874, ('bosserhof', 'accuracy'): 0.5034}
got = {('activities', 'buildings_scored'): rule_metrics['n_rows'],
       ('activities', 'precision'): round(rule_metrics['precision'], 4),
       ('activities', 'recall'): round(rule_metrics['recall'], 4),
       ('activities', 'exact_set_match'): round(rule_metrics['exact_match_rate'], 4),
       ('bosserhof', 'buildings_scored'): rule_boss_metrics['n_rows'],
       ('bosserhof', 'accuracy'): round(rule_boss_metrics['accuracy'], 4)}
print('\nREGRESSION GATE vs the published baseline')
bad = []
for k, want in BASELINE.items():
    have = got[k]
    ok = abs(have - want) < 1e-9
    print(f'  {k[0]:11s} {k[1]:18s} expected {want:>10}  got {have:>10}  {"OK" if ok else "CHANGED"}')
    if not ok:
        bad.append((k, want, have))
assert not bad, f'the rule arm no longer reproduces the published baseline: {bad}'
print('\nGate passed - the restructured scorer reproduces the published numbers.')

RULE ENGINE - ACTIVITIES
  Counting individual activity LABELS, not buildings.

  buildings scored                     : 882
  labels the human recorded  (truth)   : 1,526
  labels the engine predicted          : 1,418

    correct   (predicted AND true)     : 1,198
    extra     (predicted, NOT true)    : 220
    missing   (true, NOT predicted)    : 328

  precision = 1,198 / 1,418 = 84.5%     of the labels predicted, this share was right
  recall    = 1,198 / 1,526 = 78.5%     of the labels that exist, this share was found

  arithmetic check: 1,198 + 220 = 1,418 predicted   |  1,198 + 328 = 1,526 truth

  buildings matching the human exactly : 479 of 882 (54.3%)
    extra labels only                  : 116   (misallocates capacity)
    missing labels only                : 186   (removes capacity)
    both                               : 101

        activity  in_truth  over_predicted  missed  net_over  miss_rate
         Workers       850              21      58       -37      0.068

---
## Step 7 — Evidence tier

Graded from the **source row**, not from the classifier's own decision labels. The old
version switched on `interpreted_type in ('poi_tags', 'osm_fallback', 'no_signal')`,
which only the rule engine emits — every LLM row would land in a bucket that does not
describe it. A classifier-independent tier is what makes "who does better where"
answerable at all.

In [8]:
POI_COLS = ['amenity', 'shop', 'tourism', 'building', 'information', 'additional_information']


def evidence_tier(row):
    if any(pd.notna(row.get(c)) and str(row.get(c)).strip() not in ('', 'None')
           for c in POI_COLS):
        return '1 POI tag on the building'
    fn = str(row.get('function') or '')
    if fn in GENERIC_COMMERCIAL:
        return '2 generic commercial ALKIS code'
    if fn:
        return '3 specific ALKIS code'
    if pd.notna(row.get('osm_building_type')) or pd.notna(row.get('osm_landuse_class')):
        return '4 OSM footprint / land use'
    return '5 no usable signal'


src_tier = src.set_index('gml_id').apply(evidence_tier, axis=1).rename('tier')
print('Evidence available, over all annotated buildings:')
print(truth[['gml_id']].join(src_tier, on='gml_id')['tier']
      .value_counts().sort_index().to_string())

print('\nActivities by evidence tier, per arm (Workers included):')
rows = []
for arm, m in merged.items():
    mm = m.join(src_tier, on='gml_id')
    sel = mm['activities_scoreable'] & mm['present'].fillna(False).astype(bool)
    for tier_name, grp in mm[sel].groupby('tier'):
        pairs = [(r['gml_id'], set(r['zone_activities']), as_set(r['activities_truth']))
                 for _, r in grp.iterrows()]
        mt, _ = score_activities(pairs)
        rows.append({'arm': arm, 'tier': tier_name, 'buildings': mt['n_rows'],
                     'precision': round(mt['precision'], 3),
                     'recall': round(mt['recall'], 3),
                     'exact_set': round(mt['exact_match_rate'], 3)})
tiers = pd.DataFrame(rows)
print(tiers.pivot(index='tier', columns='arm', values='recall').to_string())

Evidence available, over all annotated buildings:


tier
1 POI tag on the building          512
2 generic commercial ALKIS code    628
3 specific ALKIS code              251

Activities by evidence tier, per arm (Workers included):


arm                              const_boss_majority  const_workers  const_workers_leisure   rule
tier                                                                                             
1 POI tag on the building                        0.0          0.491                  0.748  0.891
2 generic commercial ALKIS code                  0.0          0.688                  0.812  0.688
3 specific ALKIS code                            0.0          0.597                  0.781  0.576


---
## Step 8 — Per-arm detail and summary CSVs

`10_score_detail.csv` / `10_score_summary.csv` (unsuffixed) are the **frozen published
baseline** and are never written again. Each arm writes its own pair, and the detail
files still join on `gml_id` so any two arms can be diffed building by building.

`truth_not_producible` goes to a separate rule-diagnostics file. It counts truths that
`rule_utils`' tables cannot emit — meaningless for an open-vocabulary model, and putting
it in the comparison table would be one copy-paste away from a 870-row rule figure
sitting next to an 874-row LLM figure.

In [9]:
def join_labels(value):
    # Explicit marker for an empty set. ';'.join(set()) is '', which a CSV reader
    # restores as NaN - indistinguishable from missing data, when it actually
    # means the classifier asserted that no activity happens here.
    labels = sorted(as_set(value))
    return ';'.join(labels) if labels else '(no activity)'


for arm, m in merged.items():
    sel = m['activities_scoreable'] & m['present'].fillna(False).astype(bool)
    a = m[sel].join(src_tier, on='gml_id').copy()

    detail = a[['gml_id', 'tier', 'activities_scoreable_reason', 'label_en',
                'osm_names', 'volume_m3']].copy()
    if 'interpreted_type' in a.columns:
        detail['interpreted_type'] = a['interpreted_type']
    detail[f'bosserhof_{arm}'] = a['bosserhof_class']
    detail['bosserhof_human'] = a['bosserhof_truth']
    detail['activities_human'] = a['activities_truth'].map(join_labels)
    detail[f'activities_{arm}'] = a['zone_activities'].map(join_labels)

    mt, rws = score_activities(act_pairs(m, 'all', drop_workers=False))
    per_row = pd.DataFrame(rws).rename(columns={
        'key': 'gml_id', 'n_true_positive': 'n_correct',
        'n_over_predicted': 'n_extra', 'n_missed': 'n_missing'})
    detail = detail.merge(per_row[['gml_id', 'n_correct', 'n_extra', 'n_missing']],
                          on='gml_id', how='left')

    bm = bosserhof_accounting(boss_pairs(m, 'all'))
    d_path, s_path = score_paths(arm)
    detail.to_csv(d_path, index=False)

    correct = mt['n_true_positive']
    pd.DataFrame([
        {'dimension': 'activities', 'metric': 'buildings_scored', 'value': mt['n_rows']},
        {'dimension': 'activities', 'metric': 'labels_human',     'value': correct + mt['n_missed']},
        {'dimension': 'activities', 'metric': 'labels_predicted', 'value': correct + mt['n_over_predicted']},
        {'dimension': 'activities', 'metric': 'correct',          'value': correct},
        {'dimension': 'activities', 'metric': 'extra',            'value': mt['n_over_predicted']},
        {'dimension': 'activities', 'metric': 'missing',          'value': mt['n_missed']},
        {'dimension': 'activities', 'metric': 'precision',        'value': round(mt['precision'], 4)},
        {'dimension': 'activities', 'metric': 'recall',           'value': round(mt['recall'], 4)},
        {'dimension': 'activities', 'metric': 'exact_set_match',  'value': round(mt['exact_match_rate'], 4)},
        {'dimension': 'bosserhof',  'metric': 'buildings_scored', 'value': bm['buildings']},
        {'dimension': 'bosserhof',  'metric': 'correct',          'value': bm['correct']},
        {'dimension': 'bosserhof',  'metric': 'accuracy',         'value': round(bm['accuracy'], 4)},
    ]).to_csv(s_path, index=False)
    print(f'{arm:24s} -> {d_path.name} ({len(detail):,} rows) + {s_path.name}')

# --- rule-engine-only diagnostics ---
blocked = rule_m[(rule_m['bosserhof_truth_producible'] == False)
                 & rule_m['bosserhof_scoreable']]
keep = rule_m[(rule_m['bosserhof_truth_producible'] != False) & rule_m['bosserhof_scoreable']]
kp, _ = score_bosserhof([(r['gml_id'], r['bosserhof_class'], r['bosserhof_truth'])
                         for _, r in keep.iterrows()])
pd.DataFrame([
    {'metric': 'truth_not_producible', 'value': int(len(blocked))},
    {'metric': 'bosserhof_accuracy_excluding_blocked', 'value': round(kp['accuracy'], 4)},
    {'metric': 'bosserhof_rows_excluding_blocked', 'value': kp['n_rows']},
    {'metric': 'unreachable_classes',
     'value': len(BOSSERHOF_KNOWN - reachable_bosserhof_classes())},
    {'metric': 'source_sha256', 'value': SRC_SHA256},
]).to_csv(VALIDATION_RULE_DIAGNOSTICS, index=False)
print(f'\nrule-only diagnostics -> {VALIDATION_RULE_DIAGNOSTICS.name} '
      f'({len(blocked)} truths no rule can produce)')

rule                     -> 10_score_detail_rule.csv (882 rows) + 10_score_summary_rule.csv


const_workers            -> 10_score_detail_const_workers.csv (882 rows) + 10_score_summary_const_workers.csv


const_workers_leisure    -> 10_score_detail_const_workers_leisure.csv (882 rows) + 10_score_summary_const_workers_leisure.csv


const_boss_majority      -> 10_score_detail_const_boss_majority.csv (882 rows) + 10_score_summary_const_boss_majority.csv

rule-only diagnostics -> 10_rule_diagnostics.csv (4 truths no rule can produce)


---
## Step 9 — What the LLM said that the vocabulary has no word for

`llm_utils.validate` accepts any non-empty string as a Bosserhof class and never checks
it against the 47 known ones. Free-text drift is therefore invisible — it just scores
wrong. Reported here as a first-class finding rather than absorbed into the accuracy
number.

In [10]:
if llm_arms:
    for arm in llm_arms:
        raw = pd.read_parquet(
            VALIDATION_DIR / f'06b_llm_predictions_{arm.replace("llm_", "")}.parquet'
        )['bosserhof_class']
        resolved = raw.map(resolve_prediction_bosserhof)
        oov = resolved[resolved.notna() & (resolved != '') & ~resolved.isin(BOSSERHOF_KNOWN)]
        multi = oov[oov.map(lambda v: len(classes_mentioned(v)) >= 2)]
        print(f'{arm}: {len(oov):,} of {len(raw):,} out-of-vocabulary '
              f'({len(oov)/len(raw):.1%}), of which {len(multi):,} named 2+ classes')
        if len(oov):
            print(oov.value_counts().head(10).to_string())
        print()
else:
    print('No LLM arms scored yet.')

No LLM arms scored yet.


---
## Step 10 — How much of the "truth" is the LLM agreeing with itself

The workbook's pre-filled columns are an **earlier LLM run's answers**. Comparing this
run against them estimates self-consistency directly — which is the term the green
subset is contaminated by. Costs no API calls.

The prefill is deliberately absent from the ground-truth parquet (notebook 09 asserts
prediction columns never leak into it), so this reads the workbook.

A high agreement rate here means the green subset is measuring reproducibility, not
accuracy, and the green/red bracket should be quoted accordingly.

In [11]:
if llm_arms:
    prefill = pd.read_excel(VALIDATION_SOURCE_FILE, usecols=[0, 5], header=0,
                            names=['act_prefill', 'gml_id'])
    prefill['gml_id'] = prefill['gml_id'].astype(str)
    prefill['prefill_set'] = prefill['act_prefill'].map(
        lambda v: frozenset(x for x in str(v).replace('[', ' ').replace(']', ' ')
                            .replace("'", ' ').replace(',', ' ').split() if x))
    for arm in llm_arms:
        j = merged[arm].merge(prefill[['gml_id', 'prefill_set']], on='gml_id', how='inner')
        j = j[j['present'].fillna(False).astype(bool)]
        agree = j.apply(lambda r: set(r['zone_activities']) == set(r['prefill_set']), axis=1)
        print(f'{arm}: exact agreement with the earlier LLM run on '
              f'{int(agree.sum()):,}/{len(j):,} ({agree.mean():.1%})')
    print('\nThe green subset is an UPPER bound to the extent this number is high.')
else:
    print('No LLM arms scored yet.')

No LLM arms scored yet.


---
## Done

`10_score_comparison.csv` is the head-to-head: every arm x {activities, bosserhof} x
{all, green, red} x {Workers included, excluded}, with Wilson 95% intervals.

**Quoting these numbers responsibly:**

1. Every activity precision figure goes next to `const_workers` (0.964). Precision alone
   rewards predicting less.
2. Green and red are an upper and a lower bound for the LLM, never an estimate. The rule
   engine's figures on the same subsets *are* estimates.
3. `llm_blind` is the method comparison; `llm_full` is the deployment number. The
   difference between them is what business names are worth, not what the LLM is worth.
4. An arm gap smaller than the Wilson interval is not a gap. Bosserhof red (n=155)
   carries about +/-5.3 pp.
5. The `function` column is reconstructed from `label_en` here; production carries the
   authoritative code.